In [13]:
from groundingdino.util.inference import load_model, load_image, predict, annotate
import cv2

DEVICE = "cpu"

# model = load_model("groundingdino/config/GroundingDINO_SwinT_OGC.py", "weights/groundingdino_swint_ogc.pth", device=DEVICE)

model = load_model("groundingdino/config/GroundingDINO_SwinB_cfg.py", "weights/groundingdino_swinb_cogcoor.pth", device=DEVICE)

IMAGE_PATH = "assets/nid_45.png"
TEXT_PROMPT = "head . wing . fuselage . engine . tail"
BOX_TRESHOLD = 0.3
TEXT_TRESHOLD = 0.25

image_source, image = load_image(IMAGE_PATH)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

annotated_frame = annotate(
    image_source=image_source,
    boxes=boxes,
    logits=logits,
    phrases=phrases
)

cv2.imwrite("annotated_image_swinb.jpg", annotated_frame)
print("Saved successfully")

final text_encoder_type: bert-base-uncased
Saved successfully


# Loop Through Dataset

In [15]:
import os
import cv2
from groundingdino.util.inference import load_model, load_image, predict, annotate
from tqdm import tqdm

DEVICE = "cpu"

# Model
model = load_model(
    "groundingdino/config/GroundingDINO_SwinB_cfg.py",
    "weights/groundingdino_swinb_cogcoor.pth",
    device=DEVICE
)

# Paths
INPUT_DIR = "assets"
OUTPUT_DIR = "Annotated_Aircraft"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prompt + thresholds
TEXT_PROMPT = "airplane . wing . fuselage . engine . tail"
BOX_THRESHOLD = 0.3
TEXT_THRESHOLD = 0.25

# Loop through dataset
for img_name in tqdm(os.listdir(INPUT_DIR)):

    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    IMAGE_PATH = os.path.join(INPUT_DIR, img_name)

    # Load image
    image_source, image = load_image(IMAGE_PATH)

    # Predict
    boxes, logits, phrases = predict(
        model=model,
        image=image,
        caption=TEXT_PROMPT,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
        device=DEVICE
    )

    # Annotate
    annotated_frame = annotate(
        image_source=image_source,
        boxes=boxes,
        logits=logits,
        phrases=phrases
    )

    # Save output with same filename
    output_path = os.path.join(OUTPUT_DIR, img_name)

    cv2.imwrite(output_path, annotated_frame)

    print(f"Saved: {output_path}")

print("All images processed successfully.")

final text_encoder_type: bert-base-uncased


100%|██████████| 1/1 [00:12<00:00, 12.40s/it]

Saved: Annotated_Aircraft/nid_45.png
All images processed successfully.


# Grounding DINO + SAM (mask refinement)

In [11]:
import torch
import cv2
import numpy as np

from groundingdino.util.inference import load_model, load_image, predict
from segment_anything import sam_model_registry, SamPredictor

DEVICE = "cpu"

model = load_model("groundingdino/config/GroundingDINO_SwinT_OGC.py", "weights/groundingdino_swint_ogc.pth", device=DEVICE)
IMAGE_PATH = "assets/nid_45.png"
TEXT_PROMPT = "head . wing . fuselage . engine . tail"
BOX_TRESHOLD = 0.25
TEXT_TRESHOLD = 0.25

image_source, image = load_image(IMAGE_PATH)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

sam = sam_model_registry["vit_b"](checkpoint="weights/sam_weights/sam_vit_b.pth")
sam.to(DEVICE)

predictor = SamPredictor(sam)
predictor.set_image(image_source)

H, W, _ = image_source.shape

boxes_xyxy = boxes * torch.tensor([W, H, W, H])
boxes_xyxy = boxes_xyxy.cpu().numpy()

masks = []

for box in boxes_xyxy:
    mask, _, _ = predictor.predict(
        box=box,
        multimask_output=False
    )
    masks.append(mask[0])


annotated = image_source.copy()

for mask in masks:
    annotated[mask] = (0, 255, 0)  # green overlay

cv2.imwrite("final_sam_output.jpg", annotated)
print("Saved: final_sam_output.jpg")

final text_encoder_type: bert-base-uncased
Saved: final_sam_output.jpg
